# Experimentación con diferentes arquitecturas de modelos de clasificación de imágenes

En este cuaderno, vamos a experimentar con diferentes arquitecturas de modelos de clasificación de imágenes utilizando PyTorch. El objetivo es comparar el rendimiento de varios modelos preentrenados y determinar cuál se adapta mejor a nuestro conjunto de datos.

## Sección 1: Configuración del entorno y carga de datos

In [1]:
# Comprobación rápida del entorno
import sys
print('Python:', sys.version.split()[0])
try:
    import torch
    print('PyTorch:', torch.__version__)
    print('CUDA disponible:', torch.cuda.is_available())
except Exception as e:
    print('PyTorch no está instalado o hay un error:', e)

Python: 3.12.13
PyTorch: 2.11.0+cu128
CUDA disponible: True


In [2]:
# Imports y dispositivo
import os
import gc
import time
import itertools
import pandas as pd

import torch
from torch import nn, optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split

from torchvision import transforms, datasets
from torchvision.models import efficientnet_b3, EfficientNet_B3_Weights

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device =>', device)

Device => cuda


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# Descomprimir el archivo directamente en el entorno local de Colab
!echo "Descomprimiendo archivos..."
!unzip -q "/content/drive/MyDrive/Proyecto.zip" -d "/content/"
!echo "¡Descompresión finalizada! Los datos están listos en el disco local."

Descomprimiendo archivos...
¡Descompresión finalizada! Los datos están listos en el disco local.


In [ ]:
# Forzar la recolección de basura de Python
gc.collect()

# Vaciar la caché de la GPU
torch.cuda.empty_cache()

print(f"Memoria liberada. GPU lista para otro intento.")

Memoria liberada. GPU lista para otro intento.


In [ ]:
transformaciones_base = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

# Cargamos las imágenes
base_dir = '/content/Proyecto/data'
train_dir = os.path.join(base_dir, 'train')
dataset_base = datasets.ImageFolder(root=train_dir, transform=transformaciones_base)

# Dividimos matemáticamente (70% Train, 15% Val, 15% Test)
total_size = len(dataset_base)
train_size = int(0.7 * total_size)
val_size = int(0.15 * total_size)
test_size = total_size - train_size - val_size

train_data, val_data, test_data = random_split(
    dataset_base, [train_size, val_size, test_size]
)

# Cambiar batch_size si se tiene una diferente GPU/CPU
# Se usó 128 porque es optimo para una A100 en Colab, pero si se tiene menos memoria, se puede reducir a 64.
batch_size = 128
train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_data, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

print(f"Datos base cargados. Entrenamiento: {len(train_data)} | Validación: {len(val_data)}")

Datos base cargados. Entrenamiento: 3662 | Validación: 784


## Sección 2: Arquitectura de la red neuronal (LungX)

In [ ]:
class BackboneEfficientNetB3(nn.Module):
    def __init__(self):
        super().__init__()
        
        # Cargamos el modelo preentrenado con los pesos por defecto
        modelo_base = efficientnet_b3(weights=EfficientNet_B3_Weights.DEFAULT)
        
        # Extraemos solo la sección 'features' (las capas convolucionales)
        # Descartamos el 'classifier' final porque se hará una propia cabeza de clasificación
        self.features = modelo_base.features
        
        # Definimos los índices de las capas donde extraeremos los mapas.
        # En la arquitectura interna de EfficientNet-B3 en PyTorch, estos bloques
        # corresponden a los puntos justo después de reducir la resolución espacial.
        self.indices_extraccion = [3, 5, 7]

    def forward(self, x):
        mapas_multiescala = []
        
        # Pasamos la imagen secuencialmente capa por capa
        for i, capa in enumerate(self.features):
            x = capa(x)
            
            # Si el índice coincide con nuestras etapas clave, guardamos el tensor
            if i in self.indices_extraccion:
                mapas_multiescala.append(x)
                
            # Optimización: una vez que sacamos el tercer mapa,
            # no necesitamos seguir calculando el resto de las capas convolucionales
            if i == max(self.indices_extraccion):
                break
                
        return mapas_multiescala

In [ ]:
class ChannelAttention(nn.Module):
    def __init__(self, in_channels, ratio=16):
        super(ChannelAttention, self).__init__()
        # Reducimos la dimensión espacial a 1x1 usando average pooling y max pooling
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        
        # Perceptrón Multicapa (MLP) compartido para evaluar la importancia de los canales
        self.mlp = nn.Sequential(
            nn.Conv2d(in_channels, in_channels // ratio, 1, bias=False),
            nn.ReLU(),
            nn.Conv2d(in_channels // ratio, in_channels, 1, bias=False)
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = self.mlp(self.avg_pool(x))
        max_out = self.mlp(self.max_pool(x))
        # Se suman ambos descriptores y se normalizan entre 0 y 1
        out = avg_out + max_out
        return self.sigmoid(out)

class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super(SpatialAttention, self).__init__()
        # Una convolución para analizar los mapas comprimidos a lo largo del canal
        padding = kernel_size // 2
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=padding, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # Comprimimos la información a lo largo de los canales usando Promedio y Máximo
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        
        # Concatenamos y pasamos por la convolución
        x_cat = torch.cat([avg_out, max_out], dim=1)
        out = self.conv(x_cat)
        return self.sigmoid(out)

class CBAM(nn.Module):
    def __init__(self, in_channels, ratio=16, kernel_size=7):
        super(CBAM, self).__init__()
        self.channel_attention = ChannelAttention(in_channels, ratio)
        self.spatial_attention = SpatialAttention(kernel_size)

    def forward(self, x):
        # Se calcula la atención de canal y se multiplica por la entrada original
        out = x * self.channel_attention(x)
        # Se calcula la atención espacial usando el resultado anterior y se vuelve a multiplicar
        out = out * self.spatial_attention(out)
        return out

In [ ]:
class MultiScaleFusionBlock(nn.Module):
    def __init__(self, in_channels_list, out_channels=384, target_size=(19, 19)):
        super(MultiScaleFusionBlock, self).__init__()
        
        # Proyecciones 1x1 para igualar la profundidad de los canales
        # in_channels_list contendrá los canales reales que salen de las etapas 3, 4 y 5 de EfficientNet-B3
        self.projections = nn.ModuleList([
            nn.Conv2d(in_channels, out_channels, kernel_size=1) 
            for in_channels in in_channels_list
        ])
        
        self.target_size = target_size

    def forward(self, mapas_cbam):
        # mapas_cbam: lista de 3 tensores que ya pasaron por el CBAM
        fused_map = 0
        
        for i, mapa in enumerate(mapas_cbam):
            # Proyectar a la misma cantidad de canales 
            mapa_proyectado = self.projections[i](mapa)
            
            # Redimensionar espacialmente mediante interpolación bilineal
            mapa_redimensionado = F.interpolate(
                mapa_proyectado, 
                size=self.target_size, 
                mode='bilinear', 
                align_corners=False
            )
            
            # Suma elemento a elemento
            fused_map = fused_map + mapa_redimensionado
            
        return fused_map

In [10]:
class RedNeumonia(nn.Module):
    def __init__(self, num_classes=1, usar_cbam=True):
        super(RedNeumonia, self).__init__()
        self.usar_cbam = usar_cbam
        
        self.backbone = BackboneEfficientNetB3()
        canales = [48, 136, 384]
        
        # El interruptor arquitectónico: solo creamos atención si se solicita
        if self.usar_cbam:
            self.cbam_modules = nn.ModuleList([CBAM(c) for c in canales])
            
        self.fusion = MultiScaleFusionBlock(canales)
        self.global_pool = nn.AdaptiveAvgPool2d(1) 
        self.clasificador = nn.Sequential(
            nn.Dropout(p=0.3),
            nn.Linear(384, num_classes),
            nn.Sigmoid()
        )

    def forward(self, x):
        mapas = self.backbone(x)
        
        # Bifurcación: Evaluamos si el CBAM altera los mapas o si pasan intactos
        if self.usar_cbam:
            mapas_listos = [cbam(mapa) for cbam, mapa in zip(self.cbam_modules, mapas)]
        else:
            mapas_listos = mapas 
            
        fused = self.fusion(mapas_listos)
        pooled = self.global_pool(fused)
        pooled = pooled.view(pooled.size(0), -1)
        return self.clasificador(pooled)

## Sección 3: Funciones de perdida y motor de entrenamiento

In [11]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.8, gamma=2.0):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets):
        # inputs ya deben haber pasado por Sigmoid en la cabeza de clasificación
        bce_loss = F.binary_cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-bce_loss)
        focal_loss = self.alpha * (1 - pt) ** self.gamma * bce_loss
        return focal_loss.mean()

# Pérdida combinada del paper
def hybrid_loss(predicciones, etiquetas):
    bce = F.binary_cross_entropy(predicciones, etiquetas)
    focal = FocalLoss()(predicciones, etiquetas)
    return 0.5 * bce + 0.5 * focal

In [ ]:
def entrenar_epoca(modelo, loader, optimizador, criterio):
    modelo.train()
    perdida_total = 0.0
    
    for imagenes, etiquetas in loader:
        imagenes = imagenes.to(device)
        etiquetas = etiquetas.float().unsqueeze(1).to(device) 
        
        optimizador.zero_grad() 
        
        predicciones = modelo(imagenes)
        perdida = criterio(predicciones, etiquetas)
        
        perdida.backward() 
        optimizador.step() 
        
        perdida_total += perdida.item()
        
    return perdida_total / len(loader)

def validar_modelo(modelo, loader, criterio):
    modelo.eval()
    perdida_total = 0.0
    correctos = 0
    total = 0
    
    with torch.no_grad(): # No calculamos gradientes en validación
        for imagenes, etiquetas in loader:
            imagenes, etiquetas = imagenes.to(device), etiquetas.float().unsqueeze(1).to(device)
            
            predicciones = modelo(imagenes)
            perdida = criterio(predicciones, etiquetas)
            perdida_total += perdida.item()
            
            # Si la probabilidad es > 0.5, predecimos Neumonía (1), si no, Normal (0)
            predicciones_binarias = (predicciones > 0.5).float()
            correctos += (predicciones_binarias == etiquetas).sum().item()
            total += etiquetas.size(0)
            
    accuracy = correctos / total
    return perdida_total / len(loader), accuracy

In [ ]:
class EarlyStopping:
    def __init__(self, patience=5, min_delta=0.001):
        """
        patience: Cuántas épocas esperar sin mejoras antes de detenerse.
        min_delta: El cambio mínimo para considerar que hubo una mejora real.
        """
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_acc = None
        self.early_stop = False

    def __call__(self, val_acc):
        if self.best_acc is None:
            self.best_acc = val_acc
        elif val_acc < self.best_acc + self.min_delta:
            self.counter += 1
            print(f"    ⚠️ EarlyStopping: {self.counter}/{self.patience} épocas sin mejora significativa.")
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_acc = val_acc
            self.counter = 0 

## Sección 4: Laboratorio de evaluación de arquitecturas

In [ ]:
# --- Definir el espacio de búsqueda Arquitectónica ---
arquitecturas = [True, False] 
learning_rates = [1e-3, 5e-4]
funciones_perdida = {
    "Hibrida_Focal": hybrid_loss,
    "BCE_Pura": nn.BCELoss()
}

# --- Preparar el registro de resultados ---
max_epocas = 30 
mejor_accuracy_global = 0.0
mejor_configuracion = {}
registro_experimentos = []

print("Iniciando Evaluación de Arquitecturas...\n" + "="*70)

for usar_cbam, lr, (nombre_perdida, criterio) in itertools.product(arquitecturas, learning_rates, funciones_perdida.items()):
    nombre_arq = "Híbrida (Con Atención CBAM)" if usar_cbam else "Base (Sin Atención CBAM)"
    print(f"\n[Evaluando] Arquitectura: {nombre_arq} | LR: {lr} | Pérdida: {nombre_perdida}")
    
    modelo_exp = RedNeumonia(num_classes=1, usar_cbam=usar_cbam).to(device)
    optimizador = optim.AdamW(modelo_exp.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizador, mode='min', patience=2, factor=0.5)
    early_stopping = EarlyStopping(patience=5, min_delta=0.001)
    
    mejor_acc_local = 0.0 
    
    for epoca in range(max_epocas):
        inicio = time.time()
        
        loss_train = entrenar_epoca(modelo_exp, train_loader, optimizador, criterio)
        loss_val, acc_val = validar_modelo(modelo_exp, val_loader, criterio)
        
        scheduler.step(loss_val)
        current_lr = optimizador.param_groups[0]['lr']
        tiempo_epoca = time.time() - inicio
        
        print(f"  Época {epoca+1:02d}/{max_epocas} | Tiempo: {tiempo_epoca:.0f}s | LR: {current_lr:.6f} | Train Loss: {loss_train:.4f} | Val Loss: {loss_val:.4f} | Val Acc: {acc_val:.4f}")
        
        if acc_val > mejor_acc_local:
            mejor_acc_local = acc_val

        if acc_val > mejor_accuracy_global:
            mejor_accuracy_global = acc_val
            mejor_configuracion = {
                'arquitectura': nombre_arq,
                'lr': lr,
                'loss': nombre_perdida,
                'epoca': epoca + 1
            }
            torch.save(modelo_exp.state_dict(), "mejor_modelo_absoluto.pth")
            print("  ⭐ ¡Nuevo récord global! Arquitectura superior encontrada.")
            
        early_stopping(acc_val)
        if early_stopping.early_stop:
            print(f"  🛑 EarlyStopping: La red dejó de aprender. Finalizando prueba de esta arquitectura.")
            break
            
    registro_experimentos.append({
        "Arquitectura": nombre_arq,
        "Learning Rate": lr,
        "Función de Pérdida": nombre_perdida,
        "Mejor Val Accuracy": mejor_acc_local
    })

    del modelo_exp
    torch.cuda.empty_cache()

df_resultados = pd.DataFrame(registro_experimentos)
df_resultados.to_csv("reporte_arquitecturas.csv", index=False)

print("\n" + "="*70)
print(f"🏆 INVESTIGACIÓN FINALIZADA 🏆")
print(f"Reporte exportado a 'reporte_arquitecturas.csv'")
print(f"Mejor Accuracy de Validación: {mejor_accuracy_global:.4f}")
print(f"Conclusión de la mejor arquitectura posible:")
print(f" - Arquitectura: {mejor_configuracion['arquitectura']}")
print(f" - Optimizada con LR: {mejor_configuracion['lr']} y Función de Pérdida: {mejor_configuracion['loss']}")
print(f" - Época de convergencia: {mejor_configuracion['epoca']}")

Iniciando Evaluación de Arquitecturas...

[Evaluando] Arquitectura: Híbrida (Con Atención CBAM) | LR: 0.001 | Pérdida: Hibrida_Focal
  Época 01/30 | Tiempo: 36s | LR: 0.001000 | Train Loss: 0.0822 | Val Loss: 0.0702 | Val Acc: 0.9566
  ⭐ ¡Nuevo récord global! Arquitectura superior encontrada.
  Época 02/30 | Tiempo: 35s | LR: 0.001000 | Train Loss: 0.0300 | Val Loss: 0.0848 | Val Acc: 0.9592
  ⭐ ¡Nuevo récord global! Arquitectura superior encontrada.
  Época 03/30 | Tiempo: 35s | LR: 0.001000 | Train Loss: 0.0155 | Val Loss: 0.0384 | Val Acc: 0.9860
  ⭐ ¡Nuevo récord global! Arquitectura superior encontrada.
  Época 04/30 | Tiempo: 34s | LR: 0.001000 | Train Loss: 0.0223 | Val Loss: 0.0105 | Val Acc: 0.9923
  ⭐ ¡Nuevo récord global! Arquitectura superior encontrada.
  Época 05/30 | Tiempo: 35s | LR: 0.001000 | Train Loss: 0.0079 | Val Loss: 0.0166 | Val Acc: 0.9898
    ⚠️ EarlyStopping: 1/5 épocas sin mejora significativa.
  Época 06/30 | Tiempo: 35s | LR: 0.001000 | Train Loss: 0.0023

In [ ]:
print("Preparando el examen final para la arquitectura ganadora...\n" + "="*60)

# Recrear el modelo con la configuración exacta que ganó
modelo_ganador = RedNeumonia(num_classes=1, usar_cbam=True).to(device)

# Cargar los pesos (la "memoria") del mejor entrenamiento
modelo_ganador.load_state_dict(torch.load("mejor_modelo_absoluto.pth"))
print("✅ Pesos del modelo ganador (Época 13) cargados correctamente.")

# Crear el DataLoader para el conjunto de Test
test_loader = DataLoader(
    test_data, 
    batch_size=128, 
    shuffle=False,
    num_workers=2, 
    pin_memory=True
)

# Configurar la función de pérdida ganadora
criterio_ganador = hybrid_loss

# Ejecutar la evaluación
test_loss, test_acc = validar_modelo(modelo_ganador, test_loader, criterio_ganador)

print("\n" + "🏆 RESULTADOS DEL EXAMEN FINAL (TEST SET) 🏆")
print("="*60)
print(f"Total de imágenes evaluadas : {len(test_data)}")
print(f"Pérdida (Loss) final        : {test_loss:.4f}")
print(f"Precisión (Accuracy) final  : {test_acc:.4f} ({test_acc * 100:.2f}%)")
print("="*60)

Preparando el examen final para la arquitectura ganadora...
✅ Pesos del modelo ganador (Época 13) cargados correctamente.

🏆 RESULTADOS DEL EXAMEN FINAL (TEST SET) 🏆
Total de imágenes evaluadas : 786
Pérdida (Loss) final        : 0.0254
Precisión (Accuracy) final  : 0.9898 (98.98%)


In [ ]:
import shutil

archivos_a_guardar = [
    '/content/mejor_modelo_absoluto.pth',
    '/content/reporte_arquitecturas.csv'
]

ruta_destino_drive = '/content/drive/MyDrive/Proyecto/resultados_lungx'

os.makedirs(ruta_destino_drive, exist_ok=True)

print(f"Copiando archivos a: {ruta_destino_drive} ...")

for archivo in archivos_a_guardar:
    if os.path.exists(archivo):
        nombre_archivo = os.path.basename(archivo)
        destino_final = os.path.join(ruta_destino_drive, nombre_archivo)
        
        shutil.copy(archivo, destino_final)
        print(f" ✅ Guardado: {nombre_archivo}")
    else:
        print(f" ❌ No se encontró el archivo temporal: {archivo}")

print("\n¡Respaldo completado!")

Copiando archivos a: /content/drive/MyDrive/Proyecto/resultados_lungx ...
 ✅ Guardado: mejor_modelo_absoluto.pth
 ✅ Guardado: reporte_arquitecturas.csv

¡Respaldo completado!
